
# Accenture Project Overview

## Challenge Objective

Analyse pollutant levels in European cities using 10 years of annual air quality statistics (2015-2024) published by the European Environment Agency (EEA).
The ultimate goal of the full project is to predict future Air Quality Index (AQI) values or flag regulatory limit exceedances using machine learning.

## Dataset

- Source: EEA Annual Air Quality Statistics (e-Reporting)
- Time range: 2015 to 2024, one CSV file per year
- Pollutants covered: NO2, PM10, PM2.5, O3
- Geography: Europe-wide monitoring stations with a focus on Italy
- One row = one monitoring station x one statistical summary x one year

## Notebook Roadmap

| Part | Title | Goal |
|------|-------|------|
| 1 | Environment and Data Loading | Load all 10 CSVs into one DataFrame |
| 2 | Data Quality Assessment | Understand missingness, coverage, validation flags |
| 3 | Univariate EDA | Distribution of pollution levels per pollutant |
| 4 | Temporal EDA | Trends over 2015-2024 |
| 5 | Spatial EDA | Geographic and station-type patterns |
| 6 | Correlation and Relationships | Inter-pollutant correlations |
| 7 | Key Findings and Next Steps | Business insights and cleaning agenda |

## Note on Data Leakage

This notebook only performs exploratory analysis. No model is trained here.
Columns that encode reporting-time metadata (Data Coverage, Verification, Calculation Time) will not be used as model features in later steps.
This will be flagged explicitly each time these columns appear.

---

---
# Environment and Data Loading


10 separate CSV files, one per year between 2015-2024, loaded into a single DataFrame. After loading, we make the first inspection to understand the structure of data.

## Importing Libraries

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Force white background and light gray grid globally for PyCharm
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'grid.color': '#CCCCCC',
    'grid.alpha': 0.5
})
sns.set_style('whitegrid', rc={'figure.facecolor': 'white', 'axes.facecolor': 'white', 'grid.color': '#CCCCCC'})


# Global variables
focus_pollutants = ["NO2", "PM10", "PM2.5", "O3"]
secondary_pollutants = ["SO2", "CO", "C6H6"]
all_pollutants = focus_pollutants + secondary_pollutants

aggregation_id = "P1Y-day-per50"

print("Libraries loaded")

Libraries loaded


## Combining CSV

In [2]:
DATA = "2015-2024"
years = range(2015, 2025)

frames = []
for year in years:
    file_path = os.path.join(DATA, f"DataExtract{year}.csv")
    df_year = pd.read_csv(file_path, low_memory=False)
    frames.append(df_year)
    print(f"{year}: {df_year.shape[0]:,} rows")

df = pd.concat(frames, ignore_index=True)
print(f"\nCombined DataFrame: {df.shape[0]:,} rows, {df.shape[1]} columns")

DATA_DIR = '2015-2024'
years = list(range(2015, 2025))

2015: 36,550 rows
2016: 38,330 rows
2017: 39,012 rows
2018: 48,394 rows
2019: 50,045 rows
2020: 50,029 rows
2021: 51,287 rows
2022: 53,016 rows
2023: 50,953 rows
2024: 50,865 rows

Combined DataFrame: 468,481 rows, 27 columns


## Country comparison dataset

In [3]:
comparison = "country_comparison"
comparison_folders = ["FR", "CH", "AT"]

df_compare_list = []

for code in comparison_folders:
    folder_path = os.path.join(comparison, code)

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        df_temp = pd.read_csv(file_path, low_memory=False)
        df_compare_list.append(df_temp)
    print(f"{code} loaded.")

# Build focused dataframe directly — no need to store the raw version
df_compare_focus = pd.concat(df_compare_list, ignore_index=True)
df_compare_focus = df_compare_focus[
    (df_compare_focus["Data Aggregation Process Id"] == aggregation_id) &
    (df_compare_focus["Air Pollutant"].isin(focus_pollutants))].copy()

print("df_compare_focus rows:", f"{len(df_compare_focus):,}")
print(df_compare_focus["Country"].value_counts())


FR loaded.
CH loaded.
AT loaded.
df_compare_focus rows: 18,989
Country
France         13195
Austria         4804
Switzerland      990
Name: count, dtype: int64


## Look at the Raw Data

In [4]:
df.head(3)

,Country,Air Quality Network,Air Quality Network Name,Air Quality Station EoI Code,Air Quality Station Name,Sampling Point Id,Air Pollutant,Air Pollutant Description,Data Aggregation Process Id,Data Aggregation Process,...,Longitude,Latitude,Altitude,City,City Code,City Population,Source Of Data Flow,Calculation Time,Link to raw data (only E1a/validated data from AQ e-Reporting),Observation Frequency
0,Italy,NET.IT082A,Rete di Rilevamento della Qualit� dell'Aria de...,IT1034A,MEDA,SPO.IT1034A_5_BETA_1998-02-05_00:00:00,PM10,Particulate matter < 10 µm (aerosol),P1Y-day-per50,1 year 50 %ile of daily values in a year,...,9.1569,45.6608,221.0,Milano (greater city) ...,IT002K1 ...,3622641.0,E1a/validated data AQ e-Reporting,17/02/2026 23:01:33,https://eeadmz1batchservice02.blob.core.window...,day
1,Italy,NET.IT082A,Rete di Rilevamento della Qualit� dell'Aria de...,IT0480A,SESTO S.GIOVANNI,SPO.IT0480A_8_chemi_1974-11-01_00:00:00,NO2,Nitrogen dioxide (air),P1Y-day-min,1 year day min,...,9.2364,45.5339,137.0,Milano (greater city) ...,IT002K1 ...,3622641.0,E1a/validated data AQ e-Reporting,17/02/2026 23:01:25,https://eeadmz1batchservice02.blob.core.window...,hour
2,Italy,NET.IT082A,Rete di Rilevamento della Qualit� dell'Aria de...,IT0480A,SESTO S.GIOVANNI,SPO.IT0480A_8_chemi_1974-11-01_00:00:00,NO2,Nitrogen dioxide (air),P1Y-day-per50,1 year 50 %ile of daily values in a year,...,9.2364,45.5339,137.0,Milano (greater city) ...,IT002K1 ...,3622641.0,E1a/validated data AQ e-Reporting,17/02/2026 23:01:33,https://eeadmz1batchservice02.blob.core.window...,hour


In [5]:
# Column names and data types
print(df.dtypes)

Country                                                               str
Air Quality Network                                                   str
Air Quality Network Name                                              str
Air Quality Station EoI Code                                          str
Air Quality Station Name                                              str
Sampling Point Id                                                     str
Air Pollutant                                                         str
Air Pollutant Description                                             str
Data Aggregation Process Id                                           str
Data Aggregation Process                                              str
Year                                                                int64
Air Pollution Level                                               float64
Unit Of Air Pollution Level                                           str
Data Coverage                         

In [6]:
# Basic descriptive statistics
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Country,468481,1,Italy,468481,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air Quality Network,468481,74,NET.IT112A,38900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air Quality Network Name,468481,74,Regione Lazio,38900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air Quality Station EoI Code,468481,796,IT1993A,1678,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air Quality Station Name,468481,793,CENMO1,1678,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sampling Point Id,468481,5446,SPO.IT1743A_7_UV-P_2005-10-20_00:00:00,288,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air Pollutant,468481,38,NO2,116548,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air Pollutant Description,468481,38,Nitrogen dioxide (air),116548,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Aggregation Process Id,468481,59,P1Y,36388,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Aggregation Process,468481,58,Annual mean / 1 calendar year,36388,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
# Data Quality Assessment

*Why this step?*

Before drawing any conclusions, we need to understand how complete and trustworthy the data is.
Data quality problems — missing values, incomplete year coverage, unvalidated entries — can produce misleading charts and wrong conclusions if ignored.
This section answers: can we trust this data, and where are the gaps?

Note on leakage: Data Coverage, Verification, and Calculation Time are reporting-time metadata.
They would not be available at prediction time, so they must never be used as model features.
We study them here only to assess data quality.

## Missing Values

In [7]:
# Count missing values per column
missing_count = df.isnull().sum()
miss_percentage = (missing_count / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing_count, 'Missing %': miss_percentage})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Columns with missing values:')
print(missing_df)
missing_df.to_csv("images/missing_df.csv")

Columns with missing values:
                                                    Missing Count  Missing %
City                                                       290697      62.05
City Code                                                  290697      62.05
City Population                                            290697      62.05
Air Pollution Level                                          4629       0.99
Observation Frequency                                        1564       0.33
Link to raw data (only E1a/validated data from ...            897       0.19
Altitude                                                      149       0.03


In [15]:
# Bar chart of missing values — Plotly
import plotly.graph_objects as go

fig = go.Figure(go.Bar(
    x=missing_df['Missing %'].values,
    y=missing_df.index.tolist(),
    orientation='h',
    marker_color='steelblue'
))
fig.add_vline(x=5, line_dash='dash', line_color='red',
              annotation_text='5% threshold', annotation_position='top right')
fig.update_layout(
    title='Missing Values by Column',
    xaxis_title='Missing (%)',
    yaxis_title='',
    height=400,
    width=900,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(gridcolor='#CCCCCC'),
    yaxis=dict(gridcolor='#CCCCCC')
)
fig.write_image('images/missing_value_barchart.png', scale=2)
fig.show()

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


City is missing for ~62% of records. For Italy, we enrich station metadata by spatially joining station coordinates (lat/lon) with ISTAT ‘comuni’ boundaries using GeoPandas. This assigns each station to a municipality (Comune), which we use as City.

## Handling Missing Cities

In [ ]:
# Handling Missing cities
import geopandas as gpd

comuni = gpd.read_file("data_boundaries/comuni_2024/comuni_2024.shp")
print(comuni.crs)
print(comuni.columns.tolist())

# Convert comuni to WGS84 lat/lon to correctly match station coordinates
comuni = comuni.to_crs("EPSG:4326")
print("New CRS:", comuni.crs)

In [ ]:
# Keep Italy rows with valid coordinates
df_it = df[df["Country"] == "Italy"].copy()
df_it = df_it.dropna(subset=["Longitude", "Latitude"])

# Create geometry column from lon/lat
gdf_stations = gpd.GeoDataFrame(
    df_it,
    geometry=gpd.points_from_xy(df_it["Longitude"], df_it["Latitude"]),
    crs="EPSG:4326"
)

print("Station CRS:", gdf_stations.crs)
print("Number of Italy station rows:", len(gdf_stations))

In [ ]:
joined = gpd.sjoin(
    gdf_stations[["Air Quality Station EoI Code", "geometry"]],
    comuni[["COMUNE", "PRO_COM", "geometry"]],
    how="left",
    predicate="within"
)

print("Joined rows:", len(joined))
print("Share of stations matched to a comune:",
      round(joined["COMUNE"].notna().mean() * 100, 2), "%")

joined.head(5).to_csv("images/joined_comune.csv")
joined.head(5)

In [ ]:
# Build mapping from station code to comune name
station_to_city = (
    joined.dropna(subset=["COMUNE"])
    .groupby("Air Quality Station EoI Code")["COMUNE"]
    .agg(lambda s: s.mode().iloc[0])
)

print("Stations with assigned comune:", len(station_to_city))

In [ ]:
# Clean City column first
df["City"] = df["City"].astype("string").str.strip()
df.loc[df["City"] == "", "City"] = pd.NA

it_mask = df["Country"] == "Italy"

before = df.loc[it_mask, "City"].isna().mean() * 100

df.loc[it_mask, "City"] = df.loc[it_mask, "City"].fillna(
    df.loc[it_mask, "Air Quality Station EoI Code"].map(station_to_city)
)

after = df.loc[it_mask, "City"].isna().mean() * 100

print(f"Italy missing City — before: {before:.2f}% | after: {after:.2f}%")
print("Unique cities in full df:", df["City"].nunique())

## Population Check

Before analysing data coverage, we inspect the original EEA `City Population` field.

For the Italian subset, this variable is often missing and, when available, usually remains unchanged across years for the same city. This makes it unsuitable for studying how exposure evolves over time, which is why we later merge an external yearly population dataset.

In [ ]:
# Check whether the original EEA City Population changes over time
# Focus on Italy because the later population merge is done for Italy

pop_check = df[
    (df["Country"] == "Italy") &
    df["City"].notna() &
    df["Year"].notna()
].copy()

pop_check["City"] = pop_check["City"].astype(str).str.strip()

# One representative population value per City-Year
city_year_pop = (
    pop_check.dropna(subset=["City Population"])
    .groupby(["City", "Year"])["City Population"]
    .median()
    .reset_index()
)

# For each city, count how many distinct population values appear across years
city_pop_variation = (
    city_year_pop.groupby("City")["City Population"]
    .nunique()
    .reset_index(name="n_unique_population_values")
    .sort_values(["n_unique_population_values", "City"])
)

n_cities = len(city_pop_variation)
n_constant = (city_pop_variation["n_unique_population_values"] == 1).sum()
share_constant = n_constant / n_cities if n_cities > 0 else np.nan

print(f"Italian cities with non-missing EEA population info: {n_cities}")
print(f"Cities where EEA population stays exactly the same across all available years: {n_constant}")
print(f"Share of constant-population cities: {share_constant:.1%}")

display(city_pop_variation.head(15))
display(city_pop_variation["n_unique_population_values"].value_counts().sort_index())

In [ ]:
# Show a few example cities
example_cities = ["Roma", "Milano", "Napoli", "Torino", "Bologna"]

example_df = city_year_pop[city_year_pop["City"].isin(example_cities)].copy()

plt.figure(figsize=(8, 4.5))
for city in sorted(example_df["City"].unique()):
    sub = example_df[example_df["City"] == city].sort_values("Year")
    plt.plot(sub["Year"], sub["City Population"], marker="o", label=city)

plt.title("EEA City Population by year: example Italian cities")
plt.xlabel("Year")
plt.ylabel("City Population")
plt.legend()
plt.tight_layout()
plt.savefig("images/eea_population_examples.png", dpi=300)
plt.show()

Because the EEA city population field is mostly static across years, we next enrich the dataset with an external yearly population source to analyse demographic exposure more reliably.

## Population dataset joining

#### UN Population Dataset (2015–2024)

The EEA "City Population" field is often missing or constant over time, so we cannot study how exposure varies with population or compute per‑capita metrics reliably.

We load city-level population data from UN statistics (`UNdata_Export_20260312_103717571.csv`) for Italian cities from the United Nations' Statistical Yearbooks (https://data.un.org/Data.aspx?q=city+population&d=POP&f=tableCode%3a240#POP).

We use the UN World Urbanization Prospects city population dataset (Italy, by city and year): load it, normalize city names for matching (case‑insensitive), and fill missing years with the smaller of previous/next year (conservative for concentration per person). Then we merge with EEA city–year data and format city names for display (each city starting with a capital letter).

In [ ]:
UN_POP_PATH = "UNdata_Export_20260312_103717571.csv"

un_raw = pd.read_csv(UN_POP_PATH)
un_pop = (
    un_raw.loc[
        (un_raw["Country or Area"] == "Italy") & (un_raw["Sex"] == "Both Sexes"),
        ["Year", "City", "Value"],
    ]
    .rename(columns={"Value": "Population_UN"})
    .copy()
)
un_pop["Year"] = un_pop["Year"].astype(int)

def normalize_city_for_merge(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower()

un_pop["City_norm"] = un_pop["City"].apply(normalize_city_for_merge)

years_full = list(range(2015, 2025))
filled = []
for (city_norm, city_orig), grp in un_pop.groupby(["City_norm", "City"]):
    by_year = grp.set_index("Year")["Population_UN"]
    years_in_grp = sorted(by_year.index)
    for y in years_full:
        if y in by_year.index:
            filled.append({"City_norm": city_norm, "City": city_orig, "Year": y, "Population_UN": by_year.loc[y]})
        else:
            prev_y = [vy for vy in years_in_grp if vy < y]
            next_y = [vy for vy in years_in_grp if vy > y]
            prev_val = by_year.loc[prev_y[-1]] if prev_y else np.nan
            next_val = by_year.loc[next_y[0]] if next_y else np.nan
            if pd.notna(prev_val) and pd.notna(next_val):
                pop_fill = min(prev_val, next_val)
            elif pd.notna(prev_val):
                pop_fill = prev_val
            elif pd.notna(next_val):
                pop_fill = next_val
            else:
                continue
            filled.append({"City_norm": city_norm, "City": city_orig, "Year": y, "Population_UN": pop_fill})

un_pop_filled = pd.DataFrame(filled)
print("UN population (filled): years", sorted(un_pop_filled["Year"].unique()))

In [ ]:
print("UN population (filled): years", sorted(un_pop_filled["Year"].unique()))
firenze_pop = (
    un_pop_filled[un_pop_filled["City_norm"] == "milano"]
    [["Year", "City", "Population_UN"]]
    .sort_values("Year")
)
print('Example: population in Milano over 2015-2024')
display(firenze_pop)

### Merge with EEA city–year and city display

We build one row per (City, Year) from EEA (median pollutant, one Lat/Lon per city), merge on normalized city name and year with UN population, then apply **title case** to city names so every city starts with a capital letter in maps and tables.

In [ ]:
aggregation_id = "P1Y-day-per50"
focus_pollutants = ["NO2", "PM10", "PM2.5", "O3"]
secondary_pollutants = ["SO2", "CO", "C6H6"]
all_pollutants = focus_pollutants + secondary_pollutants

df_focus = df[
    (df["Data Aggregation Process Id"] == aggregation_id) &
    (df["Air Pollutant"].isin(focus_pollutants))
].copy()

df_copy = df.copy()


In [ ]:
coords = (
    df_focus.dropna(subset=["City", "Latitude", "Longitude"])
    .groupby("City")[["Latitude", "Longitude"]]
    .mean()
    .reset_index()
)

# ONE ROW PER City–Year–Pollutant (median annual level)
city_year_pol = (
    df_focus.dropna(subset=["City"])
    .groupby(["City", "Year", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .reset_index()
)

# Pivot pollutants into columns
city_pivot = city_year_pol.pivot_table(
    index=["City", "Year"],
    columns="Air Pollutant",
    values="Air Pollution Level",
    aggfunc="median",
).reset_index()

pol_cols = [p for p in focus_pollutants if p in city_pivot.columns]

by_city_year = city_pivot.merge(coords, on="City", how="left")
by_city_year["City_norm"] = by_city_year["City"].apply(normalize_city_for_merge)
by_city_year["Year"] = by_city_year["Year"].astype(int)

merged = by_city_year.merge(
    un_pop_filled[["City_norm", "Year", "Population_UN"]],
    on=["City_norm", "Year"],
    how="left",
)

if "City Population" in df_focus.columns:
    pop_eea = (
        df_focus.dropna(subset=["City"])
        .groupby(["City", "Year"])["City Population"]
        .first()
        .reset_index()
        .rename(columns={"City Population": "Population_EEA"})
    )
    pop_eea["Year"] = pop_eea["Year"].astype(int)
    merged = merged.merge(pop_eea, on=["City", "Year"], how="left")
    merged["City_Population"] = merged["Population_UN"].fillna(merged["Population_EEA"])
else:
    merged["City_Population"] = merged["Population_UN"]

def city_display(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower().title()

merged["City"] = merged["City"].apply(city_display)

print("Merged city-year shape:", merged.shape)
print('For example, merged data for Rome')

display(merged[merged["City"] == "Roma"].sort_values("Year"))

## Data Coverage

The Data Coverage column tells us what fraction of the year a station was actually measuring.
A station with 40% coverage was only active for about 5 months — its annual summary is less reliable.
The EEA recommends at least 75% coverage for a statistic to be considered valid.

In [ ]:
# Distribution of Data Coverage
df['Data Coverage'].dropna().plot(kind='hist', bins=40, figsize=(9, 4), color='steelblue', edgecolor='white')
plt.axvline(75, color='red', linestyle='--', label='75% threshold')
plt.xlabel('Data Coverage (%)')
plt.title('Distribution of Data Coverage')
plt.legend()
plt.tight_layout()
plt.savefig("images/distribution_of_data_coverage.png", dpi=300)
plt.show()

above_75 = (df['Data Coverage'] >= 75).sum()
below_75 = (df['Data Coverage'] < 75).sum()
print(f'Coverage >= 75%: {above_75:} rows ({above_75 / len(df) * 100:.1f}%)')
print(f'Coverage <  75%: {below_75:} rows ({below_75 / len(df) * 100:.1f}%)')

## Verification Flags

The Verification column records whether data was officially validated by the reporting authority.
Value 1 means validated. Anything else should be treated with caution.

In [ ]:
verif_counts = df['Verification'].value_counts(dropna=False)
print(verif_counts)

verif_counts.plot(kind='bar', figsize=(6, 4), color='steelblue', edgecolor='white')
plt.title('Verification Status of Records')
plt.xlabel('Verification Code')
plt.ylabel('Number of Records')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/verification_status.png", dpi=300)
plt.show()

## Aggregation Process Breakdown

Each row does not represent a direct sensor reading. It represents a statistical summary of an entire year.
Different rows for the same station and year represent different statistics:
annual median, annual maximum, annual 99th percentile, and so on.

This matters a lot: mixing different aggregation types in one analysis would be like comparing apples and oranges.
For the EDA sections that follow, we will consistently use the annual median (P1Y-day-per50),
which is the 50th percentile of daily values over a full year.

In [ ]:
agg_counts = df['Data Aggregation Process'].value_counts().head(15)
agg_counts.to_csv("images/top_15_aggregation_processes.csv")
print(agg_counts.to_string())

agg_counts.plot(kind='barh', figsize=(11, 5), color='steelblue')
plt.xlabel('Number of Records')
plt.title('Top 15 Aggregation Process Types in the Dataset')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("images/top_15_aggregation_processes.png", dpi=300)
plt.show()

## Duplicate Check

In [ ]:
# Exact duplicate rows
n_dup = df.duplicated().sum()
print(f'Exact duplicate rows: {n_dup:,} ({n_dup / len(df) * 100:.2f}%)')

# Logical duplicates: same station + pollutant + aggregation type + year
key_cols = ['Air Quality Station EoI Code', 'Air Pollutant', 'Data Aggregation Process', 'Year']
n_logical = df.duplicated(subset=key_cols).sum()
print(f'Logical duplicate rows (same station, pollutant, aggregation, year): {n_logical:,}')

## Summary

Key data quality observations — these will inform the cleaning phase (Step 3 of the challenge):

- Which columns have missing values and at what proportion
- What fraction of rows have low data coverage (below 75%) — these produce less reliable annual statistics
- How many records are unverified — these may need to be excluded from modeling
- The dataset contains multiple aggregation types per station — analysis must group by type or fix one type

---

---
# EDA: Pollutant Distributions

*Why this step?*

Before comparing across time or geography, we need to understand what each pollutant looks like on its own.
Univariate analysis answers:

- Which pollutants are most commonly measured in the dataset?
- What is the typical range of pollution levels across European stations?
- Are there extreme outliers? Is the distribution skewed?

From here onward we work with annual median values only (aggregation code P1Y-day-per50).
This gives us one representative number per station per year, making comparisons fair and consistent.

We are still on raw data — no quality filters applied. What you see includes all coverage levels and verification statuses.

## Which pollutants are present?

In [ ]:
pollutant_counts = df['Air Pollutant'].value_counts()
pollutant_counts.to_csv('images/pollutant_counts.csv')
print(pollutant_counts.to_string())

# Plotly bar chart
fig = go.Figure(go.Bar(
    x=pollutant_counts.index.tolist(),
    y=pollutant_counts.values,
    marker_color='steelblue',
    marker_line_color='white',
    marker_line_width=0.5
))
fig.update_layout(
    title='Number of Records per Pollutant (all years, all aggregation types)',
    xaxis_title='Pollutant',
    yaxis_title='Records',
    xaxis_tickangle=-30,
    height=400,
    width=900,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(gridcolor='#CCCCCC'),
    yaxis=dict(gridcolor='#CCCCCC')
)
fig.write_image('images/pollutant_counts.png', scale=2)
fig.show()

The dataset contains a large number of pollutants, but their observation frequency varies widely.

For the main analysis we focus on the four most consistently monitored pollutants (NO2, PM10, PM2.5, O3).

However, three additional pollutants (SO2, CO, C6H6) are included to enrich the interpretation of emission sources.

## Focused DataFrame

In [ ]:
# Create df_focus: EDA subset (annual median-of-daily values + main pollutants)
df_focus = df[
    (df["Data Aggregation Process Id"] == aggregation_id) &
    (df["Air Pollutant"].isin(focus_pollutants))].copy()

# Create df_extended: includes both main and secondary pollutants
df_extended = df[
    (df["Data Aggregation Process Id"] == aggregation_id) &
    (df["Air Pollutant"].isin(all_pollutants))
]

print("df_focus rows:", f"{len(df_focus):,}")
print("Countries in focus:", df_focus["Country"].nunique())
print(df_focus["Country"].value_counts())

print("\nPollutant summary (Air Pollution Level):")
print(df_focus.groupby("Air Pollutant")["Air Pollution Level"].describe().round(2))

In [ ]:
# Checking unit of air pollution for chosen pollutants
pollutant_units = (df_extended[['Air Pollutant', 'Unit Of Air Pollution Level']]
                   .drop_duplicates())            # remove duplicates)

display(pollutant_units)
pollutant_units.to_csv("images/pollutant_units.csv")

## Histograms per pollutant

In [ ]:
# Histograms — one per pollutant — Plotly
colors_plotly = ['steelblue', 'tomato', 'goldenrod', 'mediumseagreen']

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{p} — Distribution of Annual Median' for p in focus_pollutants])

for idx, (pollutant, color) in enumerate(zip(focus_pollutants, colors_plotly)):
    row, col = divmod(idx, 2)
    data = df_focus[df_focus['Air Pollutant'] == pollutant]['Air Pollution Level'].dropna()
    cap = data.quantile(0.99)
    data_capped = data[data <= cap]
    median_val = data.median()
    mean_val = data.mean()

    fig.add_trace(
        go.Histogram(x=data_capped, nbinsx=50, marker_color=color,
                     marker_line_color='white', marker_line_width=0.5,
                     name=pollutant, showlegend=False),
        row=row+1, col=col+1
    )
    fig.add_vline(x=median_val, line_dash='dash', line_color='black',
                  annotation_text=f'Median: {median_val:.1f}',
                  row=row+1, col=col+1)
    fig.add_vline(x=mean_val, line_dash='dot', line_color='red',
                  annotation_text=f'Mean: {mean_val:.1f}',
                  row=row+1, col=col+1)

fig.update_layout(
    title_text='Pollution Level Distributions (Annual Median, raw data)',
    height=700, width=1100,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_xaxes(gridcolor='#CCCCCC')
fig.update_yaxes(gridcolor='#CCCCCC')
fig.write_image('images/pollution_level_distributions.png', scale=2)
fig.show()

### Extended Pollutant Distributions (Secondary Pollutants)

In [ ]:
extra_colors = ['slateblue', 'darkorange', 'teal']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, pollutant, color in zip(axes, secondary_pollutants, extra_colors):
    data = df_extended[df_extended["Air Pollutant"] == pollutant]["Air Pollution Level"].dropna()

    cap = data.quantile(0.99)
    data_capped = data[data <= cap]

    # compute statistics
    mean_val = data_capped.mean()
    median_val = data_capped.median()

    ax.hist(data_capped, bins=30, color=color, edgecolor='white')

    # mean and median lines
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f"Mean: {mean_val:.2f}")
    ax.axvline(median_val, color='black', linestyle='-', linewidth=2, label=f"Median: {median_val:.2f}")

    unit = df_extended[df_extended["Air Pollutant"] == pollutant]["Unit Of Air Pollution Level"].dropna().mode()
    unit_text = unit.iloc[0] if not unit.empty else ""
    ax.set_title(f"{pollutant} ({unit_text})")
    ax.set_xlabel("Air Pollution Level")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.tight_layout()
plt.savefig("images/secondary_pollution_level_distributions.png", dpi=300)
plt.show()

## Boxplots

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(data=df_focus, x='Air Pollutant', y='Air Pollution Level',
            hue='Air Pollutant', legend=False,
            order=focus_pollutants, showfliers=True, flierprops={'marker':'.'},
            palette=['#4c72b0', '#55a868', '#c44e52', '#8172b2'])

plt.title('Overall Distribution of Pollutant Levels (2015-2024)')
plt.ylabel('ug/m3')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("images/overall_pollution_level_distributions.png", dpi=300)
plt.show()

## Conclusion: Business Insight

Key takeaways from the pollutant distributions:

- All distributions are right-skewed. Most stations record moderate levels, but a small number of stations are severe hotspots far above the average.
- The gap between median and mean reveals how much those extreme stations pull the average upward.
- Outliers typically represent high-traffic urban cores or industrial areas — the stations regulators care most about.
- This right skew confirms that in the cleaning step, we will need a strategy for handling extreme values before modeling.

---

---
# EDA: Trends Over 2015-2024

*Why this step?*

Temporal analysis answers the most fundamental policy question: is air quality improving over time?
Three events make this decade interesting:

- COVID-19 lockdowns (2020): traffic and industrial activity nearly stopped — did pollution drop?
- EU Clean Air directives: ongoing regulatory pressure should appear as a gradual downward trend
- Post-COVID rebound (2021-2022): did pollutant levels return once restrictions lifted?

We compute the median pollution level across all stations for each year, which gives a Europe-wide signal.

## Median Polution Level

In [ ]:
# Median pollution level by year for each pollutant
trend = (
    df_focus
    .groupby(['Year', 'Air Pollutant'])['Air Pollution Level']
    .median()
    .reset_index()
)
trend.columns = ['Year', 'Pollutant', 'Median Level']

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flatten()

for i, (pollutant, color) in enumerate(zip(focus_pollutants, colors)):
    sub = trend[trend['Pollutant'] == pollutant].sort_values('Year')
    axes[i].plot(sub['Year'], sub['Median Level'], marker='o', color=color, linewidth=2)
    axes[i].axvspan(2019.5, 2020.5, color='lightcoral', alpha=0.3, label='COVID 2020')
    axes[i].set_title(f'{pollutant} — Annual Median across All Stations')
    axes[i].set_ylabel('ug/m3')
    axes[i].set_xticks(sub['Year'].unique())
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(fontsize=9)

plt.suptitle('Temporal Trends in Air Pollution — Europe 2015-2024', fontsize=13)
plt.tight_layout()
plt.savefig("images/temporal_trend_distributions.png", dpi=300)
plt.show()

## Yearly Percentage Change

In [ ]:
YoY = trend.copy()
YoY['YoY Change %'] = YoY.groupby('Pollutant')['Median Level'].pct_change() * 100

# Plotly line chart
colors_plotly = ["steelblue", "tomato", "goldenrod", "mediumseagreen"]

fig = go.Figure()

# COVID shaded band
fig.add_vrect(
    x0=2019.5, x1=2020.5,
    fillcolor="lightcoral", opacity=0.25,
    layer="below", line_width=0,
    annotation_text="COVID 2020",
    annotation_position="top left"
)

# Zero baseline
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)

for pollutant, color in zip(focus_pollutants, colors_plotly):
    sub = YoY[(YoY['Pollutant'] == pollutant) & YoY['YoY Change %'].notna()]
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub['YoY Change %'],
        mode="lines+markers",
        marker=dict(size=7),
        line=dict(color=color, width=2),
        name=pollutant
    ))

fig.update_layout(
    title="Year-over-Year Change in Median Pollution Level (%)",
    xaxis_title="Year",
    yaxis_title="Change (%)",
    height=500, width=1100,
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(gridcolor="#CCCCCC", dtick=1),
    yaxis=dict(gridcolor="#CCCCCC"),
    legend=dict(title="Pollutant")
)
fig.write_image("images/YoY_change_in_median.png", scale=2)
fig.show()

# Print the table
(YoY.pivot(index='Year', columns='Pollutant', values='YoY Change %').round(2)
            .to_csv('images/YoY_change_in_median.csv'))
print(YoY.pivot(index='Year', columns='Pollutant', values='YoY Change %').round(2))

Since there is no 2014 data, it's impossible to calculate 2015 change

## Secondary Pollutant Trends

In [ ]:
colors_secondary = {
    "SO2": "slateblue",
    "CO": "darkorange",
    "C6H6": "teal"
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

for ax, pollutant in zip(axes, secondary_pollutants):

    subset = (
        df_extended[df_extended["Air Pollutant"] == pollutant]
        .groupby("Year")["Air Pollution Level"]
        .median()
        .reset_index()
        .sort_values("Year")
    )

    ax.plot(
        subset["Year"],
        subset["Air Pollution Level"],
        marker="o",
        linewidth=2,
        color=colors_secondary[pollutant]
    )

    unit = (
        df_extended[df_extended["Air Pollutant"] == pollutant]
        ["Unit Of Air Pollution Level"]
        .dropna()
        .mode()
    )

    unit_text = unit.iloc[0] if len(unit) > 0 else ""

    ax.set_title(f"{pollutant} yearly median")
    ax.set_xlabel("Year")
    ax.set_ylabel(unit_text)
    ax.grid(alpha=0.3)

plt.suptitle("Trends of Secondary Pollutants (2015–2024)", fontsize=14)
plt.tight_layout()
plt.savefig("images/YoY_secondary_pollutans_median.png", dpi=300)
plt.show()

secondary_summary = (
    df_extended[df_extended["Air Pollutant"].isin(secondary_pollutants)]
    .groupby("Air Pollutant")["Air Pollution Level"]
    .median()
    .round(2)
)
secondary_summary.to_csv("images/YoY_secondary_pollutans_median.csv")
secondary_summary

## Conclusion: Business Insight

Key temporal findings:

- A visible drop in 2020 for traffic-related pollutants (NO2, PM) would confirm the COVID lockdown effect — strong evidence that human activity directly drives pollution.
- A consistent downward trend from 2015 onward would support the effectiveness of EU Clean Air policies.
- If pollution rebounded sharply in 2021-2022, that tells us the structural drivers (vehicles, industry) were not fundamentally changed by the pandemic.
- The 2022-2024 trend is the most policy-relevant: is progress continuing, or has it stalled?
- Secondary pollutants (SO₂, CO, C6H6) show relatively stable or gradually declining trends, suggesting that industrial combustion and fuel-related emissions have improved steadily over the last decade, though at a slower pace than traffic-related pollutants.

---

---
# EDA: Geographic Patterns

*Why this step?*

Air quality is deeply geographic. The same pollutant can vary enormously across countries, cities, and even within a city depending on where the monitoring station is located.

Three things drive this spatial variation:

- Country-level policy: some countries enforce stricter emission rules than others
- Station placement: a station next to a motorway always reads higher than one in a park
- Urban versus rural: cities concentrate vehicles and industry; rural areas typically have cleaner air

Spatial analysis tells us where the problem is worst, which determines where modeling efforts should focus.

## Bordering countries by median pollution level per pollutant

In [ ]:
# Comparison between Italy, Switzerland, Austria, France
combined = pd.concat([df_focus, df_compare_focus], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (pollutant, color) in enumerate(zip(focus_pollutants, colors)):
    sub = combined[combined['Air Pollutant'] == pollutant]
    country_median = (
        sub.groupby('Country')['Air Pollution Level']
        .median()
        .sort_values(ascending=False)
        .head(15)
    )
    country_median.plot(kind='bar', ax=axes[i], color=color, edgecolor='white')
    axes[i].set_title(f'{pollutant} — Top 15 Countries by Median Level')
    axes[i].set_ylabel('ug/m3')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Average Pollution by Country (Annual Median, 2015-2024)', fontsize=13)
plt.tight_layout()
plt.savefig("images/average_pollution_by_country.png", dpi=300)
plt.show()


## Top polluted cities

In [ ]:
print("Unique cities in full df:", df["City"].nunique())
print("Unique cities in focus:", df_focus["City"].nunique())

In [ ]:
# Most Polluted Cities per Pollutant — Plotly with North/South colour annotation
# Latitude > 43 => North Italy, else South Italy

colors_map = {'NO2': 'steelblue', 'PM10': 'tomato', 'PM2.5': 'goldenrod', 'O3': 'mediumseagreen'}

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{p} — Top 15 Most Polluted Cities' for p in focus_pollutants])

# Build a latitude lookup: median latitude per city
_lat_lookup = (
    df_focus[df_focus['City'].notna()]
    .groupby('City')['Latitude']
    .median()
)

for idx, pollutant in enumerate(focus_pollutants):
    row, col = divmod(idx, 2)
    sub = df_focus[
        (df_focus['Air Pollutant'] == pollutant) &
        (df_focus['City'].notna())
    ]
    city_stats = sub.groupby('City')['Air Pollution Level'].agg(['median', 'count'])
    city_stats = city_stats[city_stats['count'] >= 5]
    city_median = city_stats['median'].sort_values(ascending=False).head(15)

    # Assign North/South colour per city
    bar_colors = [
        '#4c72b0' if _lat_lookup.get(city, 43) > 43 else '#dd8452'
        for city in city_median.index
    ]

    fig.add_trace(
        go.Bar(
            y=city_median.index.tolist(),
            x=city_median.values,
            orientation='h',
            marker_color=bar_colors,
            name=pollutant,
            showlegend=False
        ),
        row=row+1, col=col+1
    )

# Add dummy traces for legend (North / South)
for label, color in [('North (lat > 43°)', '#4c72b0'), ('South (lat ≤ 43°)', '#dd8452')]:
    fig.add_trace(go.Bar(x=[None], y=[None], orientation='h',
                         marker_color=color, name=label, showlegend=True))

fig.update_layout(
    title_text='Most Polluted Cities per Pollutant (Annual Median, 2015-2024) — Blue=North, Orange=South',
    height=900, width=1200,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Italian macro-area')
)
fig.update_xaxes(title_text='ug/m3', gridcolor='#CCCCCC')
fig.update_yaxes(autorange='reversed', gridcolor='#CCCCCC')
fig.write_image('images/Most polluted cities per polluntant.png', scale=2)
fig.show()

## Dynamic city-level maps

To better understand spatial and temporal pollution dynamics, we aggregate station-level measurements to the city–year level. For each city and year, pollutant concentrations are computed as the median across available monitoring stations.

This aggregated dataset is then enriched with external population data, allowing us to visualise how pollution patterns evolve across Italian cities over time.

#### Map 1: Limit‑weighted pollution index based on city/year/pollutant distribution

Colour = mean over pollutants of (concentration / EU limit). Values &gt; 1 mean the city–year exceeds at least one limit on average. Circle size = population. Years 2015–2024.

In [ ]:
# ═══════════════════════════════════════════════════════
# EEA 2024 official AQI breakpoints & helper constants
# ═══════════════════════════════════════════════════════

EU_LIMITS  = {"PM2.5": 25, "PM10": 40, "NO2": 40, "O3": 120}
WHO_GUIDE  = {"PM2.5": 5,  "PM10": 15, "NO2": 10, "O3": 60}

EEA_BANDS = {
    "PM2.5": [5, 15, 50, 90, 140],
    "PM10":  [15, 45, 120, 195, 270],
    "NO2":   [10, 25, 60, 100, 150],
    "O3":    [60, 100, 120, 160, 180],
}

BAND_NAMES  = ["Good", "Fair", "Moderate", "Poor", "Very poor", "Extremely poor"]
BAND_COLORS = ["#50F0E6", "#50CCAA", "#F0E641", "#FF5050", "#960032", "#7D2181"]

def eea_band(pollutant, conc):
    """Return 1-6 band index for a concentration value."""
    if pd.isna(conc):
        return np.nan
    thresholds = EEA_BANDS.get(pollutant)
    if thresholds is None:
        return np.nan
    for i, t in enumerate(thresholds):
        if conc <= t:
            return i + 1
    return 6

def overall_aqi(row, pollutants):
    """Overall AQI = worst (max) sub-index among available pollutants."""
    vals = [eea_band(p, row.get(p, np.nan)) for p in pollutants]
    vals = [v for v in vals if not pd.isna(v)]
    return max(vals) if vals else np.nan

def dominant_pollutant(row, pollutants):
    """Return the name of the pollutant that drives the AQI."""
    worst_band = -1
    worst_pol  = np.nan
    for p in pollutants:
        b = eea_band(p, row.get(p, np.nan))
        if not pd.isna(b) and b > worst_band:
            worst_band = b
            worst_pol  = p
    return worst_pol

print("EEA 2024 helpers loaded ✓")

In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

map1_df = merged.copy()

for p in pol_cols:
    lim = EU_LIMITS.get(p)
    if lim and p in map1_df.columns:
        map1_df[f"ratio_{p}"] = map1_df[p] / lim

ratio_cols = [c for c in map1_df.columns if c.startswith("ratio_")]
map1_df["Weighted_index"] = map1_df[ratio_cols].mean(axis=1)

map1_df = map1_df.dropna(subset=["Latitude", "Longitude", "Weighted_index", "City_Population"])

fig1 = px.scatter_mapbox(
    map1_df,
    lat="Latitude",
    lon="Longitude",
    size="City_Population",
    color="Weighted_index",
    animation_frame="Year",
    animation_group="City",
    hover_data=["City", "City_Population"] + pol_cols,
    mapbox_style="open-street-map",
    zoom=5,
    center=dict(lat=42.8, lon=12.5),
    title="Map 1: Limit-weighted pollution index (2015–2024)",
    size_max=40,
    range_color=(0, 1)
)

fig1.update_layout(
    template="plotly_white",
    height=900,
    width=1200,
    margin={"r": 0, "l": 0, "t": 50, "b": 0},
    coloraxis_colorbar_title="Weighted index<br>(mean conc/limit)",
    annotations=[
        dict(
            x=0.02, y=0.98, xref="paper", yref="paper",
            text="<b>Map 1.</b> Colour = limit-weighted index. Size = population. Index > 1 = above EU limits on average.",
            showarrow=False,
            font=dict(size=11),
            align="left",
            bgcolor="rgba(255,255,255,0.8)",
            borderpad=6,
        )
    ],
)

fig1.show()

## Station type and area comparison

In [ ]:
# Station Context — Type and Area — Plotly grouped bar

type_median = (
    df_focus.groupby(['Air Quality Station Type', 'Air Pollutant'])['Air Pollution Level']
    .median().reset_index()
)
type_pivot = type_median.pivot(index='Air Quality Station Type',
                                columns='Air Pollutant',
                                values='Air Pollution Level')[focus_pollutants]

area_median = (
    df_focus.groupby(['Air Quality Station Area', 'Air Pollutant'])['Air Pollution Level']
    .median().reset_index()
)
area_pivot = area_median.pivot(index='Air Quality Station Area',
                                columns='Air Pollutant',
                                values='Air Pollution Level')[focus_pollutants]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Median Level by Station Type',
                                    'Median Level by Station Area'])

pollutant_colors = {'NO2': '#4c72b0', 'PM10': '#55a868', 'PM2.5': '#c44e52', 'O3': '#8172b2'}

for p in focus_pollutants:
    fig.add_trace(
        go.Bar(name=p, x=type_pivot.index.tolist(), y=type_pivot[p].values,
               marker_color=pollutant_colors[p], showlegend=True),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(name=p, x=area_pivot.index.tolist(), y=area_pivot[p].values,
               marker_color=pollutant_colors[p], showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    title_text='Station Context — Type and Area Comparison',
    barmode='group',
    height=500, width=1100,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Pollutant')
)
fig.update_xaxes(tickangle=-30, gridcolor='#CCCCCC')
fig.update_yaxes(title_text='ug/m3', gridcolor='#CCCCCC')
fig.write_image('images/station_context.png', scale=2)
fig.show()

## Secondary Pollutants by Station Context

In [ ]:
# Secondary pollutants by station context
secondary_subset = df_extended[
    df_extended["Air Pollutant"].isin(secondary_pollutants)
].copy()

# Median pollution by station type
type_median_ext = (
    secondary_subset
    .groupby(["Air Quality Station Type", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .reset_index()
)

type_pivot_ext = type_median_ext.pivot(
    index="Air Quality Station Type",
    columns="Air Pollutant",
    values="Air Pollution Level"
)

# Median pollution by station area
area_median_ext = (
    secondary_subset
    .groupby(["Air Quality Station Area", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .reset_index()
)

area_pivot_ext = area_median_ext.pivot(
    index="Air Quality Station Area",
    columns="Air Pollutant",
    values="Air Pollution Level"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot by station type
type_pivot_ext[secondary_pollutants].plot(
    kind="bar",
    ax=axes[0],
    edgecolor="black"
)
axes[0].set_title("Secondary Pollutants by Station Type")
axes[0].set_ylabel("Median Pollution Level")
axes[0].set_xlabel("Station Type")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# Plot by station area
area_pivot_ext[secondary_pollutants].plot(
    kind="bar",
    ax=axes[1],
    edgecolor="black"
)
axes[1].set_title("Secondary Pollutants by Station Area")
axes[1].set_ylabel("Median Pollution Level")
axes[1].set_xlabel("Station Area")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("images/secondary_by_station.png", dpi=300)
plt.show()

**Note: CO concentrations are measured in mg/m³ while SO₂ and C6H6 are measured in µg/m³. The comparison focuses on relative differences across station contexts rather than absolute magnitudes.**

## Conclusion

Key spatial findings:

- Pollution is not evenly distributed. A small number of countries and cities drive the European average significantly upward.
- Traffic stations consistently record higher NO2 than Background stations. This directly reflects vehicle emissions and validates the monitoring approach.
- Urban areas show systematically higher levels than suburban or rural ones. This is the scientific basis for urban clean air zones.
- The city ranking gives us a hotspot map of where the air quality problem is concentrated — these cities should be the priority for both policy action and model validation.
- Secondary pollutants vary across monitoring contexts. Industrial stations show the highest SO₂ levels, while traffic and urban areas display higher CO and C6H6 concentrations. These patterns support their links to industrial combustion and vehicle emissions, helping identify potential emission sources for future modeling.




## Exploring the North–South Pollution Gradient in Italy

In [ ]:
# Italy-only subset with valid coordinates
df_it_focus = df_focus[
    (df_focus["Country"] == "Italy") &
    (df_focus["Latitude"].notna()) &
    (df_focus["Longitude"].notna()) &
    (df_focus["Air Pollution Level"].notna())
].copy()

print("Rows used for Italy North-South comparison:", f"{len(df_it_focus):,}")
print("Pollutants included:", df_it_focus["Air Pollutant"].nunique())
print(df_it_focus["Air Pollutant"].value_counts())

In [ ]:
# Split Italy into macro-areas using a fixed geographic threshold
df_it_focus["Italy_area"] = np.where(
    df_it_focus["Latitude"] > 43,
    "North",
    "South"
)

display(
    df_it_focus[
        ["City", "Latitude", "Air Pollutant", "Air Pollution Level", "Italy_area"]
    ].head(10)
)

In [ ]:
# Build a comparable composite pollution measure
# Raw pollutant levels cannot be averaged directly because pollutants have different scales.
# We therefore normalize each pollutant by its EU reference limit.

EU_LIMITS = {
    "PM10": 40,
    "PM2.5": 25,
    "NO2": 40,
    "O3": 120
}

df_it_focus["Pollution_ratio"] = (
    df_it_focus["Air Pollution Level"] /
    df_it_focus["Air Pollutant"].map(EU_LIMITS)
)

# Cap very extreme normalized values to reduce distortion (Maybe will delete it)
cap_ratio = df_it_focus["Pollution_ratio"].quantile(0.999)
df_it_focus["Pollution_ratio_capped"] = df_it_focus["Pollution_ratio"].clip(upper=cap_ratio)

print(f"99.9th percentile cap for normalized ratio: {cap_ratio:.2f}")

In [ ]:
# Summary statistics for North vs South
north_south_index = (
    df_it_focus
    .groupby("Italy_area")["Pollution_ratio_capped"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
    .sort_values("mean", ascending=False)
)

display(north_south_index)
north_south_index.to_csv("images/italy_north_south_composite_index.csv")

In [ ]:
# Visual comparison of normalized pollution
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=df_it_focus,
    x="Italy_area",
    y="Pollution_ratio_capped",
    hue="Italy_area",
    legend=False,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 4},
    palette=["#4c72b0", "#dd8452"]
)

plt.title("North vs South Italy — Composite Pollution Index")
plt.xlabel("Italian macro-area")
plt.ylabel("Pollution / EU limit")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("images/north_south_composite_pollution.png", dpi=300)
plt.show()

In [ ]:
# Pollutant-level comparison for transparency
north_south_by_pollutant = (
    df_it_focus
    .groupby(["Air Pollutant", "Italy_area"])["Air Pollution Level"]
    .median()
    .unstack()
    .reindex(focus_pollutants)
    .round(2)
)

display(north_south_by_pollutant)
north_south_by_pollutant.to_csv("images/north_south_pollutant_medians.csv")

In [ ]:
# Clear textual output
north_mean = north_south_index.loc["North", "mean"]
south_mean = north_south_index.loc["South", "mean"]
north_median = north_south_index.loc["North", "median"]
south_median = north_south_index.loc["South", "median"]

print(f"Mean composite pollution index (North): {north_mean:.3f}")
print(f"Mean composite pollution index (South): {south_mean:.3f}")
print(f"Median composite pollution index (North): {north_median:.3f}")
print(f"Median composite pollution index (South): {south_median:.3f}")

if north_median > south_median:
    print("\nNorthern Italy appears more polluted overall.")
elif south_median > north_median:
    print("\nSouthern Italy appears more polluted overall.")
else:
    print("\nNorth and South show very similar overall pollution levels.")

## Extending the Spatial Analysis: Environmental Drivers

The geographic exploration revealed that air pollution is not randomly distributed across space.
Instead, clear spatial patterns emerge across countries, cities, and monitoring station contexts.

Several results from the previous analysis motivate a deeper investigation into environmental variables:

- **Country comparisons** showed systematic differences in pollutant levels across neighboring countries, suggesting that geographic and environmental factors may influence air quality beyond local emissions.
- **City-level rankings** revealed persistent pollution hotspots, many of which are located in dense urban or industrial areas.
- **Station context analysis** demonstrated that traffic and urban stations consistently report higher pollutant concentrations than rural or background stations.

These findings indicate that **urban structure and geographic characteristics** likely play an important role in explaining pollution variability.

### Hypothesis: Environmental Context Matters

Two environmental variables are particularly relevant for understanding spatial differences in pollution:

- **Elevation**
  Higher elevations typically experience stronger wind circulation and atmospheric mixing, which can help disperse pollutants.
  In contrast, low-altitude areas—especially valleys or basins—can trap polluted air and lead to higher concentrations.

- **Green Space Ratio**
  Vegetation can influence air quality by:
  - capturing particulate matter on leaf surfaces
  - promoting airflow and cooling effects
  - reducing urban heat island intensity

Cities with higher proportions of parks, forests, or vegetated land may therefore experience lower pollution levels.

### Motivation from the North–South Comparison

A preliminary comparison between **Northern and Southern regions** shows that northern areas tend to exhibit higher pollution levels.

This pattern is consistent with known environmental conditions in Italy and central Europe:

- Northern regions contain **large industrial and urban clusters**
- The **Po Valley**, in particular, is characterized by **low elevation and limited atmospheric circulation**
- Southern regions generally have **higher terrain variation and more open landscapes**

These observations suggest that **topography and land cover** may partially explain the spatial variation observed in the dataset.

### Next Step

To explore these hypotheses, we incorporate **Green space ratio** around stations or municipalities

These variables will allow us to test whether environmental context helps explain pollution differences across regions and cities, and whether they improve predictive modeling in later stages of the project.

# Feature Engineering

## Exploring Altitude Data

In [ ]:
# Build PM2.5 dataframe using station altitude directly
df_pm25_alt = df_focus[
    (df_focus["Air Pollutant"] == "PM2.5") &
    (df_focus["Altitude"].notna())
    ].copy()

print("PM2.5 rows with station altitude:", f"{len(df_pm25_alt):,}")
print("Unique stations:", df_pm25_alt["Air Quality Station EoI Code"].nunique())
print("Years covered:", sorted(df_pm25_alt["Year"].dropna().unique().tolist()))

display(
    df_pm25_alt[
        ["City", "Year", "Air Pollution Level", "Altitude"]
    ].head(10)
)

In [ ]:
# PM2.5 vs Station Altitude — Plotly scatter + OLS regression line

import numpy as np
from scipy import stats as _stats

slope, intercept, *_ = _stats.linregress(
    df_pm25_alt['Altitude'], df_pm25_alt['Air Pollution Level']
)
x_line = np.linspace(df_pm25_alt['Altitude'].min(), df_pm25_alt['Altitude'].max(), 200)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_pm25_alt['Altitude'], y=df_pm25_alt['Air Pollution Level'],
    mode='markers',
    marker=dict(color='steelblue', opacity=0.25, size=5),
    name='Station'
))
fig.add_trace(go.Scatter(
    x=x_line, y=slope*x_line + intercept,
    mode='lines',
    line=dict(color='red', width=2),
    name='OLS fit'
))
fig.update_layout(
    title='PM2.5 vs Station Altitude',
    xaxis_title='Station altitude (m above sea level)',
    yaxis_title='PM2.5 annual median (µg/m³)',
    height=500, width=800,
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(gridcolor='#CCCCCC'),
    yaxis=dict(gridcolor='#CCCCCC')
)
fig.write_image('images/PM25_vs_altitude.png', scale=2)
fig.show()

In [ ]:
# Correlation comparison: station altitude vs municipal elevation

corr_pm25_alt = df_pm25_alt[["Air Pollution Level", "Altitude"]].corr().iloc[0, 1]

print(f"Correlation between PM2.5 and station altitude: {corr_pm25_alt:.3f}")


In [ ]:
# Elevation quartiles using station altitude
df_pm25_alt["altitude_band"] = pd.qcut(
    df_pm25_alt["Altitude"],
    q=4,
    labels=["Q1: Lowest", "Q2", "Q3", "Q4: Highest"]
)

display(
    df_pm25_alt[["City", "Altitude", "altitude_band"]]
    .drop_duplicates()
    .head(10)
)

# Boxplot PM2.5 by altitude band
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df_pm25_alt,
    x="altitude_band",
    y="Air Pollution Level",
    hue="altitude_band",
    legend=False,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 4},
    palette="Blues"
)

plt.title("PM2.5 Distribution by Station Altitude Quartile")
plt.xlabel("Altitude quartile")
plt.ylabel("PM2.5 annual median (µg/m³)")
plt.tight_layout()
plt.savefig("images/pm25_by_station_altitude_quartile.png", dpi=300)
plt.show()

## Interpretation
The correlation between PM2.5 levels and station altitude is −0.315, indicating a moderate negative relationship. Stations located at higher altitudes tend to record lower PM2.5 concentrations on average.

This pattern is consistent with known atmospheric dynamics. Low-altitude areas—particularly valleys and basins—often experience weaker air circulation and temperature inversions, which can trap polluted air near the surface and lead to higher particulate concentrations. In contrast, higher-altitude locations generally benefit from stronger atmospheric mixing and wind dispersion, which helps dilute pollutants.

## Calculating Population Density

In [ ]:
# Compute city-level population density
# Primary source:  UN export (city population + comuni area)
# Fallback source: EEA 'City Population' field + comuni area

# ── comuni area lookup (shared by both sources) ──
_comuni_area = comuni[["COMUNE", "Shape_Area"]].copy()
_comuni_area["City_norm"] = _comuni_area["COMUNE"].apply(normalize_city_for_merge)
_comuni_area["City_Area_km2"] = (_comuni_area["Shape_Area"] / 1e6).round(1)

# ── PRIMARY: UN data, most recent year per city ──
_latest = int(un_pop_filled["Year"].max())
_un = (
    un_pop_filled[un_pop_filled["Year"] == _latest]
    [["City", "City_norm", "Population_UN"]]
    .drop_duplicates(subset="City_norm")
    .rename(columns={"Population_UN": "City_Population"})
    .merge(_comuni_area[["City_norm", "City_Area_km2"]], on="City_norm", how="left")
    .dropna(subset=["City_Area_km2"])
)
_un["Pop_Density"] = (_un["City_Population"] / _un["City_Area_km2"]).round(0)

# ── FALLBACK: EEA City Population + comuni area ──
import re
_eea = (
    df[["City", "City Population"]]
    .dropna()
    .drop_duplicates(subset="City")
    .rename(columns={"City Population": "City_Population"})
)
_eea["City_norm"] = (
    _eea["City"].str.strip()
    .str.lower()
    .str.replace(r"\s*\(.*?\)", "", regex=True)
    .str.strip()
)
_eea = (
    _eea
    .merge(_comuni_area[["City_norm", "City_Area_km2"]], on="City_norm", how="left")
    .dropna(subset=["City_Area_km2"])
)
_eea["Pop_Density"] = (_eea["City_Population"] / _eea["City_Area_km2"]).round(0)

# ── COMBINE: UN primary, EEA fallback for cities not in UN ──
_un_cities = set(_un["City_norm"])
_eea_extra = _eea[~_eea["City_norm"].isin(_un_cities)]

pop_density_cities = (
    pd.concat([_un, _eea_extra], ignore_index=True)
    [["City", "City_norm", "City_Population", "City_Area_km2", "Pop_Density"]]
)

print(f"Cities with density — UN: {len(_un)}, EEA fallback added: {len(_eea_extra)}, total: {len(pop_density_cities)}")
display(
    pop_density_cities[["City", "City_Population", "City_Area_km2", "Pop_Density"]]
    .sort_values("Pop_Density", ascending=False)
    .reset_index(drop=True)
)

## Integrating Green Space Ratio

In [ ]:
# ── CORINE Green Space Ratio ──
df_green = pd.read_csv("LandCover2018/green_ratio.csv")

# Create Italy dataframe first
df_focus_it_geo = df_focus[df_focus["Country"] == "Italy"].copy()

df_focus_it_geo = df_focus_it_geo.merge(
    df_green[["Air Quality Station EoI Code", "green_ratio"]],
    on="Air Quality Station EoI Code",
    how="left"
)

print("Rows after merge:", len(df_focus_it_geo))
print("Missing green_ratio:", df_focus_it_geo["green_ratio"].isna().sum())
df_focus_it_geo[["Air Quality Station EoI Code", "green_ratio"]].head(5)

In [ ]:
# Correlation
corr = (
    df_focus_it_geo[df_focus_it_geo["Air Pollutant"] == "PM2.5"]
    [["Air Pollution Level", "green_ratio"]]
    .corr().iloc[0, 1]
)
print(f"Correlation between green_ratio and PM2.5: {corr:.3f}")
print("Negative = greener comuni tend to have lower PM2.5 — expected direction")

In [ ]:
# Green Ratio vs PM2.5: scatter + boxplot — Plotly
# (named 'population density and human activity' in the chart list)

pm25_per_station = (
    df_focus_it_geo[df_focus_it_geo['Air Pollutant'] == 'PM2.5']
    .groupby('Air Quality Station EoI Code')['Air Pollution Level']
    .median().reset_index()
    .rename(columns={'Air Pollution Level': 'pm25_median'})
)

plot_df = pm25_per_station.merge(
    df_focus_it_geo[['Air Quality Station EoI Code', 'green_ratio']].drop_duplicates(),
    on='Air Quality Station EoI Code', how='inner'
).dropna()

plot_df['green_band'] = pd.qcut(
    plot_df['green_ratio'], q=4,
    labels=['Q1 Most urban', 'Q2', 'Q3', 'Q4 Most green']
)

print(f'Stations used: {len(plot_df)}')

import numpy as np
from scipy import stats as _stats

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=[
                        'PM2.5 vs Green Space Ratio (per station, Italy 2015–2024)',
                        'PM2.5 by Green Space Quartile (per station, Italy 2015–2024)'
                    ])

# Left: scatter with OLS regression line
fig.add_trace(
    go.Scatter(x=plot_df['green_ratio'], y=plot_df['pm25_median'],
               mode='markers',
               marker=dict(color='steelblue', opacity=0.4, size=6),
               name='Station', showlegend=False),
    row=1, col=1
)
slope, intercept, *_ = _stats.linregress(plot_df['green_ratio'], plot_df['pm25_median'])
x_line = np.linspace(plot_df['green_ratio'].min(), plot_df['green_ratio'].max(), 100)
fig.add_trace(
    go.Scatter(x=x_line, y=slope*x_line + intercept,
               mode='lines', line=dict(color='red', width=2),
               name='OLS fit', showlegend=False),
    row=1, col=1
)

# Right: boxplot by quartile
band_palette = {'Q1 Most urban': '#e05c5c', 'Q2': '#f5a623',
                'Q3': '#a8d5a2', 'Q4 Most green': '#4caf50'}
for band in ['Q1 Most urban', 'Q2', 'Q3', 'Q4 Most green']:
    vals = plot_df[plot_df['green_band'] == band]['pm25_median']
    fig.add_trace(
        go.Box(y=vals, name=band, marker_color=band_palette[band],
               boxpoints='outliers', showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    title_text='Population Density & Human Activity — Green Space vs PM2.5',
    height=500, width=1200,
    plot_bgcolor='white', paper_bgcolor='white'
)
fig.update_xaxes(gridcolor='#CCCCCC')
fig.update_yaxes(gridcolor='#CCCCCC')
fig.update_xaxes(title_text='Green Ratio (0=fully urban, 1=fully green)', row=1, col=1)
fig.update_yaxes(title_text='Median PM2.5 (µg/m³)', row=1, col=1)
fig.update_xaxes(title_text='Green Space Band', row=1, col=2)
fig.update_yaxes(title_text='Median PM2.5 (µg/m³)', row=1, col=2)
fig.write_image('images/population density and human activity.png', scale=2)
fig.show()

corr = plot_df[['green_ratio', 'pm25_median']].corr().iloc[0, 1]
print(f'\nCorrelation green_ratio vs PM2.5: {corr:.3f}')

**Does the greenness of a municipality explain the pollution level of the station located there?**
Green space ratio is a static property of a place — it does not change year to year in our data. Correlating it against 10 repeated
annual measurements per station would artificially inflate the sample size and give each station unequal weight.

By taking the **median PM2.5 across all years per station**, we get one representative value per location — which is the right unit of comparison when the predictor variable (green_ratio) is also measured at the station level.

## Building df_model

This dataframe is the feature matrix we will use for predicting PM2.5.
Each row is one monitoring station x one year.
We start from the annual-median PM2.5 rows and add co-pollutants + station context as features.

| Feature | Source | Notes |
|---------|--------|-------|
| PM2_5 | df_focus | Target variable |
| NO2 | df_focus (pivot) | Co-feature: combustion proxy |
| PM10 | df_focus (pivot) | Co-feature: coarse particles |
| O3 | df_focus (pivot) | Co-feature: secondary pollutant |
| Altitude | df_focus | Station elevation above sea level |
| Latitude | df_focus | Station geographic position |
| Longitude | df_focus | Station geographic position |
| Station_Type | df_focus | Traffic / Background / Industrial |
| Station_Area | df_focus | Urban / Suburban / Rural |
| Green_Ratio | df_green | Fraction of green land cover in 1 km buffer |
| Population_Density | placeholder | Empty for now |
| Year | df_focus | Temporal feature |

In [ ]:
# ── Step 1: pivot the four focus pollutants so each becomes its own column ──
# One row per station x year, columns = NO2 / PM10 / PM2.5 / O3
pivot = (
    df_focus
    .groupby(["Air Quality Station EoI Code", "Year", "Air Pollutant"])["Air Pollution Level"]
    .median()
    .unstack("Air Pollutant")
    .reset_index()
)

# Keep only rows that have a PM2.5 measurement (that is our target)
pivot = pivot.dropna(subset=["PM2.5"])

# ── Step 2: pull station-level metadata (constant per station) ──
meta_cols = [
    "Air Quality Station EoI Code",
    "Altitude",
    "Latitude",
    "Longitude",
    "Air Quality Station Type",
    "Air Quality Station Area"
]
station_meta = (
    df_focus[meta_cols]
    .drop_duplicates(subset="Air Quality Station EoI Code")
)

# ── Step 3: merge pivot with station metadata ──
df_model = pivot.merge(station_meta, on="Air Quality Station EoI Code", how="left")

# ── Step 4: merge green ratio (from CORINE land cover file) ──
df_model = df_model.merge(
    df_green[["Air Quality Station EoI Code", "green_ratio"]],
    on="Air Quality Station EoI Code",
    how="left"
)

# ── Step 5: Population Density placeholder ──
# Left empty for now; will be filled once population data is available
_density_map = pop_density_cities.set_index("City")["Pop_Density"]

_station_city = (
    df_focus[["Air Quality Station EoI Code", "City"]]
    .drop_duplicates(subset="Air Quality Station EoI Code")
    .set_index("Air Quality Station EoI Code")["City"]
)

df_model["Population_Density"] = (
    df_model["Air Quality Station EoI Code"]
    .map(_station_city)
    .map(_density_map)
)


# ── Step 6: rename columns to clean names ──
df_model = df_model.rename(columns={
    "PM2.5": "PM2_5",
    "Air Quality Station Type": "Station_Type",
    "Air Quality Station Area": "Station_Area",
    "green_ratio": "Green_Ratio"
})

print("df_model shape:", df_model.shape)
print("\nColumn overview:")
print(df_model.dtypes)

In [ ]:
# Quick inspection of the resulting feature matrix
display(df_model.head(5))

print("\nMissing values per column:")
print(df_model.isnull().sum())

## Feature Correlation Heatmap

To check whether all selected features carry independent information, we compute pairwise Pearson correlations across all numeric columns in `df_model`.

Categorical features (`Station_Type`, `Station_Area`) are label-encoded numerically for this analysis only — they are not modified in `df_model`.

**What to look for:** features with |r| > 0.9 are nearly redundant and one can likely be dropped before modeling.

In [ ]:
# Numeric columns available in df_model
numeric_cols = ["NO2", "PM10", "O3",
                "Altitude", "Latitude", "Longitude",
                "Green_Ratio", "Population_Density", "Year", "PM2_5"]

df_corr = df_model[numeric_cols].copy()

# Label-encode categoricals for correlation purposes only
df_corr["Station_Type"] = df_model["Station_Type"].astype("category").cat.codes
df_corr["Station_Area"] = df_model["Station_Area"].astype("category").cat.codes

# Reorder: features first, target (PM2_5) last
feature_order = ["NO2", "PM10", "O3",
                 "Altitude", "Latitude", "Longitude",
                 "Station_Type", "Station_Area",
                 "Green_Ratio", "Population_Density", "Year",
                 "PM2_5"]
df_corr = df_corr[feature_order]

corr = df_corr.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={"size": 8}
)
plt.title("Feature Correlation Heatmap (df_model)", pad=14)
plt.tight_layout()
plt.savefig("images/df_model_feature_correlation.png", dpi=300)
plt.show()

---
# Correlation and Feature Relationships

*Why this step?*

Correlation analysis answers:

- Do pollutants co-occur at the same stations? If NO2 is high, is PM10 also high?
- Is there a relationship between data quality (coverage) and reported pollution levels?
- Which variables carry information useful for future modeling?

To compare pollutants per station, we reshape the data so each row is one station-year
and each pollutant becomes its own column. This only works because we fixed one aggregation type (annual median).

Note on leakage: Data Coverage is a reporting-time variable studied here only for quality assessment.

## Reshaping data

In [ ]:
# One row per station-year, pollutants as columns
pivot_corr = (
    df_focus
    .groupby(['Air Quality Station EoI Code', 'Year', 'Air Pollutant'])['Air Pollution Level']
    .median()
    .unstack('Air Pollutant')
    .reset_index()
)

# Only keep the focus pollutants that actually exist in this table
available = [p for p in focus_pollutants if p in pivot_corr.columns]

corr_matrix = pivot_corr[available].corr()
print('Correlation matrix between pollutants at the same station-year:')
print(corr_matrix.round(3))

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Pollutant Correlation Matrix (Annual Median, per Station-Year)', pad=20)
plt.tight_layout()
plt.savefig("images/correlation_matrix.png", dpi=300)
plt.show()

## Exploratory Correlation Across All Selected Pollutants

In [ ]:
# Exploratory correlation across all selected pollutants

pivot_corr_ext = (
    df_extended
    .groupby(['Air Quality Station EoI Code', 'Year', 'Air Pollutant'])['Air Pollution Level']
    .median()
    .unstack('Air Pollutant')
    .reset_index()
)

# Keep only pollutants that are actually present
available_ext = [p for p in all_pollutants if p in pivot_corr_ext.columns]

corr_matrix_ext = pivot_corr_ext[available_ext].corr()

print("Exploratory correlation matrix across all selected pollutants:")
print(corr_matrix_ext.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix_ext,
    annot=True,
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title('Exploratory Correlation Across All Selected Pollutants', pad=20)
plt.tight_layout()
plt.savefig("images/extended_correlation_matrix.png", dpi=300)
plt.show()

## Pollutant Pairs: Scatter plots

In [ ]:
pairs = [('PM10', 'PM2.5'), ('NO2', 'O3'), ('NO2', 'PM10'), ('O3', 'PM2.5')]

# Unstack pollutants into columns for seaborn
df_scatter = df_focus.pivot_table(
    index=['Air Quality Station EoI Code', 'Year'],
    columns='Air Pollutant',
    values='Air Pollution Level',
    aggfunc='median'
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, (a, b) in enumerate(pairs):
    # Drop rows where either pollutant is missing to avoid seaborn errors
    plot_df = df_scatter.dropna(subset=[a, b])
    
    sns.scatterplot(data=plot_df, x=a, y=b, alpha=0.2, s=8, color=colors[i], ax=axes[i])
    axes[i].set_title(f"{a} vs {b}")

plt.tight_layout()
plt.savefig("images/pairs_pollutants_scatterplot.png", dpi=300)
plt.show()


## Effect of Data Coverage level

In [ ]:
# Does data coverage level affect the recorded pollution value?
# (Leakage note: studied here for quality purposes only — not a future feature)

df_no2 = df_focus[df_focus['Air Pollutant'] == 'NO2'][['Data Coverage', 'Air Pollution Level']].dropna()
df_no2['Coverage Bucket'] = pd.cut(
    df_no2['Data Coverage'],
    bins=[0, 50, 75, 90, 100],
    labels=['below 50%', '50-75%', '75-90%', 'above 90%']
)

bucket_median = df_no2.groupby('Coverage Bucket', observed=True)['Air Pollution Level'].median()
bucket_median.to_csv("images/data_coverage_bucket_no2_median.csv")
print(bucket_median)

bucket_median.plot(kind='bar', figsize=(8, 4), color='steelblue', edgecolor='white')
plt.title('NO2 Median Level by Data Coverage Bucket')
plt.ylabel('Median NO2 (ug/m3)')
plt.xlabel('Data Coverage Bucket')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/data_coverage_bucket_no2_median.png", dpi=300)
plt.show()

## Conclusion: Business Insight

Key correlation findings:

- NO2 and PM10/PM2.5 tend to co-occur at the same stations. These pollutants share sources: combustion engines and industrial processes. Targeting one source (e.g., reducing car traffic) will tend to improve multiple pollutants simultaneously.
- If correlations are strong, a single feature set could be used to predict multiple pollutant targets without building entirely separate models.
- The coverage-vs-level check is a data quality signal. If stations with low coverage report systematically lower pollution, they may be missing peak episodes (e.g., winter heating). This would bias the dataset and must be addressed in the cleaning step.
- The expanded correlation analysis highlights moderate links between traffic-related pollutants (NO2, CO, C6H6), while O3 shows negative correlations with combustion pollutants, reflecting its secondary atmospheric formation.

---

---
# Key Findings and Next Steps

While the core analysis focuses on four major pollutants (NO2, PM10, PM2.5, O3), three additional pollutants (SO2, CO, and C6H6) were explored to provide further insight into emission sources.
These pollutants are associated with industrial combustion (SO2), incomplete fuel combustion from vehicles (CO), and fuel evaporation or traffic-related emissions (C6H6).
Their inclusion helps connect pollutant patterns with potential emission drivers.

---

## Data Quality Summary — Cleaning Agenda for Step 3

| Issue | What We Observed | Action Needed in Step 3 |
|-------|-----------------|-------------------------|
| Missing values | Some columns have significant gaps | Drop or impute per column |
| Low data coverage | Some stations measured less than 75% of the year | Filter or flag as low-quality |
| Unverified records | A fraction of rows are not officially validated | Filter to Verification equal to 1 |
| Multiple aggregation types | Rows mix medians, maxima, percentiles | Fix one aggregation type per analysis |
| City name padding | Trailing whitespace in City column | Apply str.strip() |
| Extreme outliers | Right-skewed distributions with very high values | Cap or log-transform before modeling |

---

## Business Insights Summary

| # | Insight | Business Implication |
|---|---------|---------------------|
| 1 | Distributions are right-skewed — most stations are moderate, a few are severe hotspots | Hotspot-targeted interventions deliver the biggest health impact per euro spent |
| 2 | Visible pollution drop in 2020 coinciding with COVID lockdowns | Human activity is the primary driver — structural change can and does reduce pollution |
| 3 | Long-term downward trend from 2015 (to be confirmed after cleaning) | EU Clean Air policies show results, but progress is uneven across countries |
| 4 | Traffic stations record significantly higher NO2 than Background stations | Vehicle emissions are the dominant urban source of NO2 |
| 5 | Urban areas consistently more polluted than rural areas | Dense cities are the priority zones for public health intervention |
| 6 | NO2, PM10, and PM2.5 are positively correlated at the same station | A single policy or model targeting combustion sources addresses multiple pollutants |

---

## Recommended Modeling Direction — Step 4

Two modeling approaches are feasible based on the EDA:

Option A — Regression: predict the annual median pollution level for a given station and year.

Option B — Classification: predict whether a station will exceed EU regulatory limits in a given year.
- NO2 annual limit: 40 ug/m3
- PM10 annual limit: 40 ug/m3
- PM2.5 annual limit: 25 ug/m3

Recommended starting point: Option B (classification). It produces a clear binary output that is directly actionable for policy reporting, and it is simpler to evaluate.

## Data Leakage Prevention for Step 4

The following columns must not be used as model features:

- Data Coverage: only available after the measurement period ends, not at prediction time
- Verification: a post-hoc validation status, not a prediction-time variable
- Calculation Time: when the statistic was computed, not a real-world predictor
- Air Pollution Level: if predicting whether a limit is exceeded, the label is derived from this column — using it as a feature would be direct leakage

---

Step 2 (Preliminary Analysis) finished.
Next: Step 3 — Data Cleaning and Transformation.